# Install all dependencies

In [1]:
!pip install transformers datasets torch scikit-learn pandas numpy

## Import all necessary libraries

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [3]:
# Load CSV
df = pd.read_csv("/content/mental_heath_dataset.csv")

print(df.head())
print(df.shape)

   Unique_ID                                               text   status
0        0.0                                         oh my gosh  Anxiety
1        1.0  trouble sleeping, confused mind, restless hear...  Anxiety
2        2.0  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3        3.0  I've shifted my focus to something else but I'...  Anxiety
4        4.0  I'm restless and restless, it's been a month n...  Anxiety
(49612, 3)


In [4]:
df.drop(columns="Unique_ID")

,text,status
0,oh my gosh,Anxiety
1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,I've shifted my focus to something else but I'...,Anxiety
4,"I'm restless and restless, it's been a month n...",Anxiety
...,...,...
49607,i can't explain it but i know that i don't wan...,Depression
49608,nobody ever told me that when i started treatm...,Depression
49609,my wife and i split up in 2012/2013. she had a...,Depression
49610,A close family member committed suicideI just ...,Suicidal


In [5]:
print(df.isnull().sum())

Unique_ID    9600
text            0
status          0
dtype: int64


In [6]:
encoder = LabelEncoder()
df['status_encoded'] = encoder.fit_transform(df['status'])

In [7]:
status_map = dict(zip(encoder.classes_, range(len(encoder.classes_))))
print(status_map)

{'Anxiety': 0, 'Depression': 1, 'Normal': 2, 'Suicidal': 3}


In [8]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['status_encoded'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['status_encoded']
)

In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [11]:
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

test_dataset = Dataset.from_dict({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask'],
    'labels': test_labels
})

In [12]:
num_classes = len(df['status_encoded'].unique())

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_classes
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
def compute_metrics(pred):
    logits, labels = pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': accuracy
    }

In [14]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy'
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.431424,0.469546,0.825053
2,0.392449,0.454787,0.841278
3,0.272654,0.583523,0.841479


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=14886, training_loss=0.38530045767777327, metrics={'train_runtime': 3328.9789, 'train_samples_per_second': 35.767, 'train_steps_per_second': 4.472, 'total_flos': 7832101647172608.0, 'train_loss': 0.38530045767777327, 'epoch': 3.0})

In [17]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.5835234522819519, 'eval_accuracy': 0.841479391313111, 'eval_runtime': 77.4083, 'eval_samples_per_second': 128.19, 'eval_steps_per_second': 16.032, 'epoch': 3.0}


In [18]:
predictions = trainer.predict(test_dataset)

pred_labels = np.argmax(predictions.predictions, axis=1)

print(classification_report(test_labels, pred_labels))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      1101
           1       0.78      0.75      0.76      2901
           2       0.95      0.95      0.95      3678
           3       0.73      0.78      0.75      2243

    accuracy                           0.84      9923
   macro avg       0.83      0.83      0.83      9923
weighted avg       0.84      0.84      0.84      9923



In [19]:
model.save_pretrained('./bert_tweet_model')
tokenizer.save_pretrained('./bert_tweet_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_tweet_model/tokenizer_config.json',
 './bert_tweet_model/tokenizer.json')

In [20]:
from transformers import BertTokenizer, BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('./bert_tweet_model')
tokenizer = BertTokenizer.from_pretrained('./bert_tweet_model')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [22]:
text = "I failed in my exam i am going to die"

inputs = tokenizer(
    text,
    return_tensors='pt',
    truncation=True,
    padding=True,
    max_length=128
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1).item()

predicted_label = encoder.inverse_transform([prediction])[0]

print("Predicted Label:", predicted_label)

Predicted Label: Suicidal


In [24]:
import shutil

# Create zip file
shutil.make_archive('bert_tweet_model', 'zip', './bert_tweet_model')

'/content/bert_tweet_model.zip'

In [25]:
shutil.make_archive('results', 'zip', './results')

'/content/results.zip'